# TEK — Kaggle constraints test

This notebook measures the Kaggle environment TEK depends on. It is not product code.

**Before running**

1. Notebook settings: Python, CPU, **Internet ON**.
2. Run the cells in order. The first cell installs QEMU and cloudflared (~2–3 min).
3. The TCG cell boots Alpine and measures boot and first-frame time.
4. The tunnel cell opens a Cloudflare quick tunnel and starts a 30 s heartbeat.
5. Three storage cells: a host block-layer benchmark, a guest virtio test, and a
   standalone export with a space check, `qemu-img check`, and a SHA-256 hash.
6. Open the tunnel hostname from another device.
7. Leave the notebook idle for **60 minutes**. Do not stop it.
8. Run the report cell, then the cleanup cell.

**What these numbers do and do not mean**

- Alpine netboot proves TCG boot, display, and virtio basics. It does **not** predict
  Windows 10 LTSC performance, and its storage numbers are not Windows storage numbers.
- The host block-layer numbers come from `qemu-img bench`, which bypasses the guest.
  The guest numbers come from `dd` inside the Alpine initramfs.
- If a storage cell cannot complete, it records `inconclusive` with the serial log. That
  is a valid result; do not replace it with an estimate.

The notebook uses no secrets. The tunnel serves a blank test page.


In [ ]:
# Cell 1 - environment facts and dependency install
import json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

TMP = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
REPORT = WORK / "tek_constraints.json"
FACTS = {
    "schema": "tek.constraints.v0",
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "facts": {},
    "benchmarks": {},
}

def sh(cmd, timeout=900, check=False):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    if check and p.returncode != 0:
        raise RuntimeError(f"{cmd}\n{out[-2000:]}")
    return p.returncode, out

def read(path, default=None):
    try:
        return Path(path).read_text().strip()
    except Exception:
        return default

def gib(n):
    return round(n / (1024 ** 3), 2)

print("euid:", os.geteuid() if hasattr(os, "geteuid") else "n/a")
print("python:", sys.version.split()[0])
print("tmp:", TMP, "work:", WORK)

# --- CPU ---
nproc = os.cpu_count() or 1
affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else nproc
cpu = {"nproc": nproc, "affinity": affinity, "cgroup_v2_cpu_max": read("/sys/fs/cgroup/cpu.max")}
if cpu["cgroup_v2_cpu_max"]:
    q, p = cpu["cgroup_v2_cpu_max"].split()[:2]
    if q != "max":
        cpu["effective"] = max(1, -(-int(q) // int(p)))
        cpu["source"] = "cgroup-v2"
if "effective" not in cpu:
    q = read("/sys/fs/cgroup/cpu/cpu.cfs_quota_us")
    p = read("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
    if q and p and int(q) > 0:
        cpu["effective"] = max(1, -(-int(q) // int(p)))
        cpu["source"] = "cgroup-v1"
if "effective" not in cpu:
    cpu["effective"] = affinity
    cpu["source"] = "affinity"
cpu["model"] = next(
    (line.split(":", 1)[1].strip() for line in read("/proc/cpuinfo", "").splitlines() if line.startswith("model name")),
    None,
)
flags = next(
    (line.split(":", 1)[1].strip() for line in read("/proc/cpuinfo", "").splitlines() if line.startswith("flags")),
    "",
)
cpu["vmx"] = " vmx " in f" {flags} "
cpu["svm"] = " svm " in f" {flags} "
FACTS["facts"]["cpu"] = cpu

# --- RAM ---
mem = {}
for line in read("/proc/meminfo", "").splitlines():
    key, value = line.split(":", 1)
    mem[key] = int(value.strip().split()[0]) * 1024
limit = None
cg_max = read("/sys/fs/cgroup/memory.max")
if cg_max and cg_max != "max":
    limit = int(cg_max)
else:
    v1 = read("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    if v1 and int(v1) < (1 << 60):
        limit = int(v1)
effective_ram = min(x for x in [limit, mem.get("MemAvailable")] if x)
FACTS["facts"]["ram"] = {
    "mem_total_gib": gib(mem.get("MemTotal", 0)),
    "mem_available_gib": gib(mem.get("MemAvailable", 0)),
    "cgroup_max": cg_max,
    "effective_gib": gib(effective_ram),
}

# --- disk ---
disks = {}
for path in ["/", "/kaggle/working", "/kaggle/temp"]:
    if os.path.exists(path):
        usage = shutil.disk_usage(path)
        disks[path] = {
            "total_gib": gib(usage.total),
            "free_gib": gib(usage.free),
            "writable": os.access(path, os.W_OK),
        }
FACTS["facts"]["disk"] = disks

# --- KVM / GPU ---
FACTS["facts"]["kvm"] = {
    "dev_kvm": os.path.exists("/dev/kvm"),
    "note": "expected absent; QEMU will use TCG",
}
rc, out = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
FACTS["facts"]["gpu"] = {
    "present": rc == 0,
    "detail": out.splitlines() if rc == 0 else None,
    "usable_by_guest": False,
}

print(json.dumps(FACTS["facts"], indent=2)[:4000])

# --- dependencies ---
if shutil.which("qemu-system-x86_64") is None:
    print("installing qemu...")
    rc, out = sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y qemu-system-x86")
    print(out[-1200:])
    FACTS["facts"]["qemu_install_rc"] = rc
FACTS["facts"]["qemu"] = shutil.which("qemu-system-x86_64")

if shutil.which("cloudflared") is None:
    print("installing cloudflared...")
    rc, out = sh("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    if rc == 0:
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    FACTS["facts"]["cloudflared_install_rc"] = rc
FACTS["facts"]["cloudflared"] = shutil.which("cloudflared")

print("qemu:", FACTS["facts"]["qemu"])
print("cloudflared:", FACTS["facts"]["cloudflared"])


In [ ]:
# Cell 2 - writable throughput and dataset mount behavior
def throughput(path, mb=512):
    target = Path(path) / "tek_write_test.bin"
    start = time.time()
    rc_w, _ = sh(f"dd if=/dev/zero of={target} bs=1M count={mb} conv=fdatasync status=none")
    write_s = time.time() - start
    start = time.time()
    rc_r, _ = sh(f"dd if={target} of=/dev/null bs=1M status=none")
    read_s = time.time() - start
    target.unlink(missing_ok=True)
    return {
        "write_mbps": round(mb / write_s, 1),
        "read_mbps": round(mb / read_s, 1),
        "rc": [rc_w, rc_r],
    }

FACTS["benchmarks"]["disk_io"] = {}
for path in ["/kaggle/temp", "/kaggle/working"]:
    if os.path.exists(path):
        try:
            FACTS["benchmarks"]["disk_io"][path] = throughput(path)
        except Exception as exc:
            FACTS["benchmarks"]["disk_io"][path] = {"error": repr(exc)}
print(json.dumps(FACTS["benchmarks"]["disk_io"], indent=2))

input_root = Path("/kaggle/input")
dataset = {
    "exists": input_root.exists(),
    "entries": [],
    "read_only": None,
    "largest_file": None,
    "read_mbps": None,
}
if input_root.exists():
    dataset["entries"] = sorted(p.name for p in input_root.iterdir())
    probe = input_root / ".tek_write_probe"
    try:
        probe.write_text("x")
        probe.unlink()
        dataset["read_only"] = False
    except Exception:
        dataset["read_only"] = True
    files = [p for p in input_root.rglob("*") if p.is_file()]
    if files:
        largest = max(files, key=lambda p: p.stat().st_size)
        size = largest.stat().st_size
        dataset["largest_file"] = {"name": str(largest), "gib": gib(size)}
        if size > 64 * 1024 * 1024:
            cap = min(size, 1 << 30)
            start = time.time()
            read_bytes = 0
            with open(largest, "rb") as handle:
                while read_bytes < cap:
                    chunk = handle.read(min(8 * 1024 * 1024, cap - read_bytes))
                    if not chunk:
                        break
                    read_bytes += len(chunk)
            elapsed = time.time() - start
            dataset["read_mbps"] = round(read_bytes / elapsed / 1e6, 1)
FACTS["facts"]["dataset_mount"] = dataset
print(json.dumps(dataset, indent=2))


## 3. TCG boot benchmark

Boots an Alpine netboot kernel under TCG and measures time to a login prompt and time
to the first non-blank frame. This measures the infrastructure, not Windows performance.


In [ ]:
# Cell 3 - TCG boot benchmark (Alpine netboot kernel)
import socket
from PIL import Image, ImageStat

netboot = "https://dl-cdn.alpinelinux.org/alpine/latest-stable/releases/x86_64/netboot/"
for name in ["vmlinuz-virt", "initramfs-virt"]:
    dest = TMP / name
    if not dest.exists():
        print("downloading", name)
        urllib.request.urlretrieve(netboot + name, dest)

serial = TMP / "tek_serial.log"
mon_path = TMP / "tek_qemu_mon"
for path in [serial, mon_path]:
    path.unlink(missing_ok=True)

sh("pkill -f qemu-system-x86_64 || true")
vcpu = min(4, max(1, FACTS["facts"]["cpu"]["effective"] - 1))
ram_mb = min(4096, max(1024, int(FACTS["facts"]["ram"]["effective_gib"] * 1024) - 2048))
qemu = (
    f"qemu-system-x86_64 -accel tcg,thread=multi -cpu max -smp {vcpu} -m {ram_mb} "
    f"-kernel {TMP / 'vmlinuz-virt'} -initrd {TMP / 'initramfs-virt'} "
    f"-append 'console=tty0 console=ttyS0' -serial file:{serial} "
    f"-vnc 127.0.0.1:1 -monitor unix:{mon_path},server,nowait -daemonize"
)
rc, out = sh(qemu)
print("qemu rc:", rc, out[-500:])

def mon(cmd):
    try:
        sock = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
        sock.settimeout(5)
        sock.connect(str(mon_path))
        sock.sendall((cmd + "\n").encode())
        time.sleep(0.3)
        try:
            data = sock.recv(65536).decode(errors="replace")
        except socket.timeout:
            data = ""
        sock.close()
        return data.strip()
    except Exception as exc:
        return f"monitor error: {exc!r}"

start = time.time()
login_at = None
frame_at = None
shot = TMP / "tek_screen.ppm"
for i in range(180):
    time.sleep(2)
    text = serial.read_text(errors="replace") if serial.exists() else ""
    if login_at is None and re.search(r"login:|Welcome to Alpine|/ #", text):
        login_at = round(time.time() - start, 1)
    if frame_at is None and i % 3 == 0:
        mon(f"screendump {shot}")
        if shot.exists():
            try:
                image = Image.open(shot).convert("L")
                if ImageStat.Stat(image).stddev[0] > 5:
                    frame_at = round(time.time() - start, 1)
            except Exception:
                pass
    if login_at and frame_at:
        break

FACTS["benchmarks"]["tcg_boot"] = {
    "guest": "alpine-netboot-virt",
    "vcpu": vcpu,
    "ram_mb": ram_mb,
    "login_seconds": login_at,
    "first_frame_seconds": frame_at,
    "serial_tail": serial.read_text(errors="replace")[-1500:] if serial.exists() else None,
    "qemu_status": mon("info status"),
}
print(json.dumps({k: v for k, v in FACTS["benchmarks"]["tcg_boot"].items() if k != "serial_tail"}, indent=2))
print("serial tail:", FACTS["benchmarks"]["tcg_boot"]["serial_tail"])
sh("pkill -f qemu-system-x86_64 || true")


## 4. Tunnel and heartbeat

Starts a blank HTTP page behind a Cloudflare quick tunnel, verifies it end-to-end five
times, and writes a heartbeat every 30 seconds. Leave the notebook idle for 60 minutes.


In [ ]:
# Cell 4 - outbound tunnel and session heartbeat
import threading
from datetime import datetime, timezone

site = TMP / "tek_site"
site.mkdir(exist_ok=True)
(site / "index.html").write_text(
    "<html><body><h1>TEK tunnel test</h1><p>server time: "
    + datetime.now(timezone.utc).isoformat()
    + "</p></body></html>"
)

sh("pkill -f 'http.server 8099' || true; pkill -f cloudflared || true")
subprocess.Popen(
    [sys.executable, "-m", "http.server", "8099", "--bind", "127.0.0.1", "--directory", str(site)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)

cf_log = TMP / "tek_cloudflared.log"
cf_log.unlink(missing_ok=True)

def start_tunnel(protocol=None):
    cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8099", "--no-autoupdate", "--logfile", str(cf_log)]
    if protocol:
        cmd += ["--protocol", protocol]
    subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(45):
        time.sleep(1)
        text = cf_log.read_text(errors="replace") if cf_log.exists() else ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if match:
            return match.group(0), protocol or "quic"
    return None, None

url, protocol = start_tunnel()
if not url:
    url, protocol = start_tunnel("http2")

FACTS["benchmarks"]["tunnel"] = {"url": url, "protocol": protocol, "self_fetch": []}
print("tunnel:", url)
if url:
    for _ in range(5):
        start = time.time()
        try:
            with urllib.request.urlopen(url, timeout=15) as response:
                FACTS["benchmarks"]["tunnel"]["self_fetch"].append(
                    {"status": response.status, "seconds": round(time.time() - start, 2)}
                )
        except Exception as exc:
            FACTS["benchmarks"]["tunnel"]["self_fetch"].append(
                {"error": repr(exc), "seconds": round(time.time() - start, 2)}
            )
        time.sleep(2)
print(json.dumps(FACTS["benchmarks"]["tunnel"], indent=2))

heartbeat = WORK / "tek_heartbeat.log"
heartbeat.write_text("")
stop = threading.Event()

def beat():
    count = 0
    while not stop.is_set():
        count += 1
        with heartbeat.open("a") as handle:
            handle.write(f"{datetime.now(timezone.utc).isoformat()} {count}\n")
            handle.flush()
        stop.wait(30)

threading.Thread(target=beat, daemon=True).start()
print("heartbeat:", heartbeat)
print("Leave the notebook idle for 60 minutes, then run the report cell.")


## 5. QCOW2 storage

Two layers, deliberately separate:

- **Host block layer** — `qemu-img bench` on a QCOW2 image: sequential 1 MiB and strided
  4 KiB, read and write, plus `qemu-img check`.
- **Guest virtio** — the same Alpine netboot kernel with `rdinit=/bin/sh` and a
  `-drive if=virtio` disk. The guest writes and reads a known pattern, then powers off;
  the host verifies `qemu-img check` and compares hashes.
- **Export safeguards** — a backed overlay is converted to a standalone image with a
  pre-flight space check, `.part` file, `qemu-img check`, SHA-256, atomic rename, and a
  check that the original overlay is unchanged afterwards.

The guest test measures the QEMU virtio + QCOW2 path under TCG. It does not measure a
Windows filesystem, NTFS behaviour, or Windows boot.


In [ ]:
# Cell 5 - QCOW2 block-layer benchmark (host side)
import hashlib
import shutil

def sha256_file(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

STORAGE = {"schema": "tek.storage.v0", "host_bench": {}, "guest_test": {}}
store = TMP / "tek_storage"
shutil.rmtree(store, ignore_errors=True)
store.mkdir(parents=True, exist_ok=True)

free_gib = shutil.disk_usage(TMP).free / (1024 ** 3)
img_gib = max(1, min(4, int(free_gib * 0.05)))
img = store / "test.qcow2"
rc, out = sh(f"qemu-img create -f qcow2 {img} {img_gib}G")
STORAGE["image"] = {
    "path": str(img),
    "virtual_gib": img_gib,
    "free_gib_at_create": round(free_gib, 2),
    "create_rc": rc,
    "create_out": out[-400:],
}
print(json.dumps(STORAGE["image"], indent=2))

def bench(label, opts, count):
    start = time.time()
    rc, out = sh(f"qemu-img bench -f qcow2 -c {count} {opts} {img}", timeout=1800)
    seconds = time.time() - start
    return {
        "rc": rc,
        "count": count,
        "seconds": round(seconds, 2),
        "us_per_op": round(seconds / count * 1e6, 1),
        "tail": out[-500:],
    }

STORAGE["host_bench"]["seq_write_1m"] = bench("seq_write_1m", "-d 32 -s 1M -t writeback --pattern=0x5a -w", 64)
STORAGE["host_bench"]["seq_read_1m"] = bench("seq_read_1m", "-d 32 -s 1M -t writeback", 64)
STORAGE["host_bench"]["strided_write_4k"] = bench("strided_write_4k", "-d 1 -s 4k -S 1M -t writeback --pattern=0x5a -w", 200)
STORAGE["host_bench"]["strided_read_4k"] = bench("strided_read_4k", "-d 1 -s 4k -S 1M -t writeback", 200)

check_rc, check_out = sh(f"qemu-img check {img}")
STORAGE["host_bench"]["check"] = {"rc": check_rc, "out": check_out[-500:]}
STORAGE["host_bench"]["sha256"] = sha256_file(img)
print(json.dumps(STORAGE["host_bench"], indent=2))


In [ ]:
# Cell 6 - QCOW2 guest virtio test (Alpine netboot, rdinit shell)
import signal
import socket

def parse_timings(text):
    pattern = r"real\s+(?:(\d+)m)?([\d.]+)s?"
    return [
        round(int(m.group(1) or 0) * 60 + float(m.group(2)), 2)
        for m in re.finditer(pattern, text)
    ]

def serial_run(sock_path, lines, ready="TEK_READY", done="TEK_DONE", timeout=600):
    buffer = ""
    deadline = time.time() + timeout
    conn = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    conn.settimeout(2)
    last_error = None
    for _ in range(30):
        try:
            conn.connect(str(sock_path))
            last_error = None
            break
        except OSError as exc:
            last_error = exc
            time.sleep(1)
    if last_error:
        raise RuntimeError(f"serial connect failed: {last_error!r}")

    def pump():
        nonlocal buffer
        try:
            chunk = conn.recv(65536)
            if chunk:
                buffer += chunk.decode(errors="replace")
        except socket.timeout:
            pass

    # Resend the probe until the shell answers: input sent before the shell
    # starts can be dropped by the tty layer.
    while time.time() < deadline and ready not in buffer:
        conn.sendall((lines[0] + "\n").encode())
        for _ in range(10):
            pump()
            time.sleep(0.5)
    for line in lines[1:]:
        conn.sendall((line + "\n").encode())
        time.sleep(0.8)
        pump()
    while time.time() < deadline and done not in buffer:
        pump()
        time.sleep(0.5)
    conn.close()
    return buffer

guest_img = store / "guest.qcow2"
pid_path = store / "guest.pid"
pid_path.unlink(missing_ok=True)
sh(f"qemu-img create -f qcow2 {guest_img} {img_gib}G")
before_hash = sha256_file(guest_img)

serial_sock = store / "guest_serial.sock"
serial_sock.unlink(missing_ok=True)

command = (
    f"qemu-system-x86_64 -accel tcg,thread=multi -cpu max -smp {vcpu} -m {ram_mb} "
    f"-kernel {TMP / 'vmlinuz-virt'} -initrd {TMP / 'initramfs-virt'} "
    f"-append 'rdinit=/bin/sh console=ttyS0' "
    f"-drive file={guest_img},if=virtio,format=qcow2,cache=writeback "
    f"-serial unix:{serial_sock},server=on,wait=off "
    f"-pidfile {pid_path} "
    f"-display none -daemonize"
)
rc, out = sh(command)
time.sleep(8)

script = [
    "echo TEK_READY",
    "mount -t devtmpfs devtmpfs /dev 2>/dev/null || true",
    "ls -l /dev/vda",
    "time dd if=/dev/zero of=/dev/vda bs=1M count=64 2>&1",
    "sync",
    "time dd if=/dev/vda of=/dev/null bs=1M count=64 2>&1",
    "dd if=/dev/urandom of=/tmp/pat bs=1M count=8 2>&1",
    "time dd if=/tmp/pat of=/dev/vda bs=1M count=8 seek=8 2>&1",
    "sync",
    "dd if=/dev/vda bs=1M count=8 skip=8 2>&1 | md5sum",
    "md5sum /tmp/pat",
    "echo TEK_DONE",
    "poweroff -f",
]
try:
    log = serial_run(serial_sock, script) if rc == 0 else ""
except Exception as exc:
    log = f"serial error: {exc!r}"

def qemu_alive():
    try:
        os.kill(int(pid_path.read_text().strip()), 0)
        return True
    except Exception:
        return False

# Give the guest time to power off cleanly before the hard kill.
for _ in range(30):
    if not qemu_alive():
        break
    time.sleep(1)
if qemu_alive():
    try:
        os.kill(int(pid_path.read_text().strip()), signal.SIGKILL)
    except Exception:
        pass
time.sleep(2)

hashes = re.findall(r"\b([0-9a-f]{32})\b", log)
timings = parse_timings(log)
check_rc, check_out = sh(f"qemu-img check {guest_img}")
match = len(hashes) >= 2 and hashes[-2] == hashes[-1]
STORAGE["guest_test"] = {
    "qemu_rc": rc,
    "qemu_out": out[-400:],
    "serial_tail": log[-2500:],
    "vda_seen": "/dev/vda" in log,
    "timings_seconds": timings,
    "seq_write_64m_s": timings[0] if len(timings) > 0 else None,
    "seq_read_64m_s": timings[1] if len(timings) > 1 else None,
    "pattern_write_8m_s": timings[2] if len(timings) > 2 else None,
    "hashes": hashes,
    "pattern_match": match,
    "check_rc": check_rc,
    "check_out": check_out[-500:],
    "sha256_before": before_hash,
    "sha256_after": sha256_file(guest_img),
    "status": (
        "ok" if (match and check_rc == 0)
        else "check_dirty" if (match and check_rc == 3)
        else "inconclusive"
    ),
}
print(json.dumps({k: v for k, v in STORAGE["guest_test"].items() if k != "serial_tail"}, indent=2))
print("serial tail:", STORAGE["guest_test"]["serial_tail"])


In [ ]:
# Cell 7 - standalone export with safeguards
export_dir = store / "export"
export_dir.mkdir(exist_ok=True)
base = store / "base.qcow2"
overlay = store / "overlay.qcow2"
final = export_dir / "session.qcow2"
part = export_dir / "session.qcow2.part"
part.unlink(missing_ok=True)
final.unlink(missing_ok=True)

sh(f"qemu-img create -f qcow2 {base} {img_gib}G")
sh(f"qemu-img bench -f qcow2 -c 32 -s 1M -t writeback --pattern=0x5a -w {base}")
sh(f"qemu-img create -f qcow2 -b {base} -F qcow2 {overlay}")
sh(f"qemu-img bench -f qcow2 -c 16 -s 1M -t writeback --pattern=0xa5 -w {overlay}")
overlay_hash_before = sha256_file(overlay)

free_bytes = shutil.disk_usage(export_dir).free
virtual_bytes = img_gib * 1024 ** 3
needed_bytes = int(virtual_bytes * 1.10)
space_ok = free_bytes >= needed_bytes

start = time.time()
if space_ok:
    convert_rc, convert_out = sh(
        f"qemu-img convert -O qcow2 -c -o compression_type=zstd {overlay} {part}",
        timeout=1800,
    )
else:
    convert_rc, convert_out = None, "skipped: insufficient free space"
convert_seconds = round(time.time() - start, 2)

if part.exists():
    check_rc, check_out = sh(f"qemu-img check {part}")
    info_rc, info_out = sh(f"qemu-img info --output=json {part}")
else:
    check_rc, check_out = None, "no .part file"
    info_rc, info_out = None, "{}"

promoted = False
if convert_rc == 0 and check_rc == 0:
    os.replace(part, final)
    promoted = True

export_hash = sha256_file(final) if final.exists() else None
overlay_preserved = overlay.exists() and sha256_file(overlay) == overlay_hash_before

info_ok = False
try:
    info = json.loads(info_out)
    info_ok = True
except Exception:
    info = {}
# A parse failure must not count as "no backing file".
backing_removed = info_ok and "backing-filename" not in info

STORAGE["export"] = {
    "virtual_gib": img_gib,
    "free_gib_before": round(free_bytes / 1024 ** 3, 2),
    "needed_gib": round(needed_bytes / 1024 ** 3, 2),
    "space_ok": space_ok,
    "convert_rc": convert_rc,
    "convert_seconds": convert_seconds,
    "convert_tail": convert_out[-400:],
    "check_rc": check_rc,
    "check_tail": check_out[-400:],
    "promoted": promoted,
    "sha256": export_hash,
    "overlay_preserved": overlay_preserved,
    "backing_removed": backing_removed,
    "status": "ok" if (
        space_ok and convert_rc == 0 and check_rc == 0
        and promoted and overlay_preserved and export_hash and backing_removed
    ) else "failed",
}
print(json.dumps(STORAGE["export"], indent=2))


## 6. Report

Summarizes the heartbeat and writes `tek_constraints.json`. Run this after the idle
period. If the session died, reopen the notebook and run it again.


In [ ]:
# Cell 8 - report
from datetime import datetime

heartbeat = WORK / "tek_heartbeat.log"
lines = [line.strip() for line in heartbeat.read_text().splitlines() if line.strip()] if heartbeat.exists() else []
FACTS["benchmarks"]["session"] = {
    "heartbeat_first": lines[0] if lines else None,
    "heartbeat_last": lines[-1] if lines else None,
    "heartbeat_count": len(lines),
    "expected_count_60min": 121,
}

gaps = []
previous = None
for line in lines:
    try:
        timestamp = datetime.fromisoformat(line.split()[0])
    except Exception:
        continue
    if previous and (timestamp - previous).total_seconds() > 90:
        gaps.append({
            "from": previous.isoformat(),
            "to": timestamp.isoformat(),
            "seconds": round((timestamp - previous).total_seconds()),
        })
    previous = timestamp
FACTS["benchmarks"]["session"]["gaps_over_90s"] = gaps

FACTS["benchmarks"]["storage"] = globals().get("STORAGE")

storage = FACTS["benchmarks"].get("storage") or {}
host = storage.get("host_bench") or {}
guest = storage.get("guest_test") or {}
tunnel_items = ((FACTS["benchmarks"].get("tunnel") or {}).get("self_fetch") or [])
export = storage.get("export") or {}
FACTS["summary"] = {
    "tcg_boot_ok": (FACTS["benchmarks"].get("tcg_boot") or {}).get("login_seconds") is not None,
    "tunnel_ok": bool(tunnel_items) and all(item.get("status") == 200 for item in tunnel_items),
    "storage_host_ok": all(
        (host.get(name) or {}).get("rc") == 0
        for name in ["seq_write_1m", "seq_read_1m", "strided_write_4k", "strided_read_4k"]
    ),
    "storage_guest_status": guest.get("status"),
    "storage_integrity_ok": guest.get("pattern_match"),
    "export_status": export.get("status"),
    "export_overlay_preserved": export.get("overlay_preserved"),
    "session_heartbeat_count": len(lines),
}

REPORT.write_text(json.dumps(FACTS, indent=2))
print(json.dumps(FACTS["summary"], indent=2))
print(json.dumps(storage, indent=2)[:4000])
print("\nreport:", REPORT)
print("\nPaste this file back for analysis. It contains no secrets.")


## 7. Cleanup

Stops the tunnel, HTTP server, and any QEMU process. Run when finished.


In [ ]:
# Cell 9 - cleanup (run when done)
event = globals().get("stop")
if event:
    event.set()
sh("pkill -f 'http.server 8099' || true; pkill -f cloudflared || true; pkill -f qemu-system-x86_64 || true")
print("stopped. heartbeat log:", WORK / "tek_heartbeat.log")
